In [1]:
import geopandas as gpd
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike
index_col = "bike_index" if network == "bike" else "walk_index"

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-3'



if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-3'

#Read the files
index = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index = index.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

gg_perimeter = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp')
gg_perimeter = gg_perimeter.to_crs(operation_crs)

iris = gpd.read_file(f'{input_file_path}/network_agreg/CONTOURS-IRIS-PE_3-0__GPKG_LAMB93_FXX_2025-01-01/CONTOURS-IRIS-PE/1_DONNEES_LIVRAISON_2025-09-00130/CONTOURS-IRIS-PE_3-0_GPKG_LAMB93_FXX-ED2025-01-01/contours-iris-pe.gpkg')
iris = iris.to_crs(operation_crs)

communes_contours = gpd.read_file(f'{input_file_path}/network_agreg/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_HOHEITSGEBIET.shp')
communes_contours = communes_contours.to_crs(operation_crs)

cantons_contours = gpd.read_file(f'{input_file_path}/network_agreg/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp')
cantons_contours = cantons_contours.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

lac_contours = gpd.read_file(f'{input_file_path}/network_agreg/GEO_LAC_LEMAN-SHP/GEO_LAC_LEMAN.shp')
lac_contours = lac_contours.to_crs(operation_crs)

import pandas as pd
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

Set parameters : GG or GE, bike or walk


**Preparer la couche Secteur Infracommunal GG**

In [ ]:
gg_geometry = gg_perimeter.union_all()
leman_contour = lac_contours[lac_contours['NOM'] == 'Léman'].copy()
gg_land_perimeter = gpd.GeoDataFrame(geometry=[gg_geometry.difference(leman_contour.union_all())], crs=operation_crs)
gg_land_geometry = gg_land_perimeter.union_all()
vaud_canton = cantons_contours[cantons_contours['NAME'] == 'Vaud'].copy()

geneva_contours = gpd.clip(zones_girec, gg_land_perimeter)[['OBJECTID', 'NOM', 'geometry']].copy()
geneva_contours['source'] = 'zones_girec'
geneva_contours['country'] = 'CH'
geneva_contours['territory'] = 'GE'
geneva_contours['unit_id'] = geneva_contours['OBJECTID'].astype(str)
geneva_contours['unit_name'] = geneva_contours['NOM']
geneva_contours['commune_name'] = geneva_contours['NOM']

iris_gg = gpd.clip(iris, gg_land_perimeter)[['code_iris', 'nom_iris', 'nom_commune', 'geometry']].copy()
iris_gg['source'] = 'iris'
iris_gg['country'] = 'FR'
iris_gg['territory'] = 'FR'
iris_gg['unit_id'] = iris_gg['code_iris'].astype(str)
iris_gg['unit_name'] = iris_gg['nom_iris']
iris_gg['commune_name'] = iris_gg['nom_commune']

vaud_communes = communes_contours[
    (communes_contours['KANTONSNUM'] == 22)
    & communes_contours.intersects(gg_land_geometry)
    & communes_contours.intersects(vaud_canton.union_all())
].copy()
vaud_communes = gpd.clip(vaud_communes, gg_land_perimeter)[['BFS_NUMMER', 'NAME', 'geometry']].copy()
vaud_communes['source'] = 'communes'
vaud_communes['country'] = 'CH'
vaud_communes['territory'] = 'VD'
vaud_communes['unit_id'] = vaud_communes['BFS_NUMMER'].astype(int).astype(str)
vaud_communes['unit_name'] = vaud_communes['NAME']
vaud_communes['commune_name'] = vaud_communes['NAME']

gg_contours = gpd.GeoDataFrame(
    pd.concat([
        geneva_contours[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
        iris_gg[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
        vaud_communes[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
    ], ignore_index=True),
    crs=operation_crs,
)

gg_infra_communal = gg_contours.copy()

print(gg_contours[['source', 'territory']].value_counts())
print(f'Total units: {len(gg_contours)}')
gg_contours.head()


**GIREC**

In [ ]:
# Spatial join
segments_girec = gpd.sjoin(index, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")

#Drop nan values 
zones_girec = zones_girec.dropna(subset=[index_col])

**GG infra-communal**

In [ ]:
# Spatial join: assign each segment to a gg infra communal unit
segments_gg_infra_communal = gpd.sjoin(index, gg_infra_communal, how="inner", predicate="within")

# Aggregate by weighted mean
gg_infra_communal_stats = (
    segments_gg_infra_communal
    .groupby("unit_id")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to gg infra communal polygons
gg_infra_communal = gg_infra_communal.merge(gg_infra_communal_stats, on="unit_id", how="left")

# Drop rows with missing values
gg_infra_communal = gg_infra_communal.dropna(subset=[index_col])

**Carreau 200**

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=[index_col])

**Export**

In [ ]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

gg_infra_communal.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_gg_infra_communal.gpkg"), driver="GPKG")
gg_infra_communal.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_gg_infra_communal.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')